# Can the coefficients of a coherent sum be recovered by minimizing the residual?

Subtracting measured inputs from an output is the time-domain form of the same problem as conditioning a spectrum. If $y = \sum_i a_i x_i + n$ and the $x_i$ are known, the $a_i$ are whatever coefficients leave the smallest residual. `optimal_linear_combination` minimizes the RMS of that residual. With no time shifts, and with the mean removed from every series, the objective in its docstring is

$$\min_c\ \mathrm{RMS}\Big[\,y + \sum_i c_i x_i\,\Big].$$

The inputs are added to the output, so the coefficient recovered is the negative of the one in the signal, $c_i = -a_i$.

The notebook knows the answer, which is what makes it a test. With one input, $y = -2.5\,x + n$ and $x$ a 1 Hz sinusoid, so the recovered coefficient is $+2.5$. With three inputs, $y = 1.5\,x_1 - 0.8\,x_2 + 2.0\,x_3 + n$, so the recovered coefficients are $-1.5$, $+0.8$, and $-2.0$. Here $x_1$ is a 1 Hz sine, $x_2$ a 0.5 Hz cosine, and $x_3$ a decaying exponential. Each record has 1000 samples over 10 s, and $n$ is white with standard deviation 0.1. The fit can trade one coefficient against another when the inputs are nearly linearly dependent over the record, and it can stop short, because the coefficients come from an iterative minimizer rather than from the normal equations.

The call is `optimal_linear_combination(df, inputs, output, domain="time")`, and the objective is the RMS of the samples. The same function with `domain="frequency"` minimizes the RMS of a Welch spectrum of the residual instead. This notebook does not take that path, and it never calls `compute_spectrum`, so no one-sided or two-sided convention is involved.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from speckit.dsp import optimal_linear_combination
from speckit.plotting import default_rc

plt.rcParams.update(default_rc)

In [ ]:
# Utility: create synthetic signals
def create_synthetic_data_1d(n=1000, noise_level=0.1, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, 10, n)
    x = np.sin(2 * np.pi * 1.0 * t)  # input
    noise = noise_level * np.random.randn(n)
    y = -2.5 * x + noise  # output = -2.5 * x + noise
    return pd.DataFrame({"x": x, "y": y})


def create_synthetic_data_miso(n=1000, noise_level=0.1, seed=0):
    np.random.seed(seed)
    t = np.linspace(0, 10, n)
    x1 = np.sin(2 * np.pi * 1.0 * t)
    x2 = np.cos(2 * np.pi * 0.5 * t)
    x3 = np.exp(-0.1 * t)
    noise = noise_level * np.random.randn(n)
    y = 1.5 * x1 - 0.8 * x2 + 2.0 * x3 + noise
    return pd.DataFrame({"x1": x1, "x2": x2, "x3": x3, "y": y})

In [ ]:
# Run 1D test
df1d = create_synthetic_data_1d()
res_1d, residual_1d = optimal_linear_combination(
    df1d, inputs=["x"], output="y", domain="time"
)

print("\nRecovered coefficient:", res_1d.x)
plt.figure()
plt.title("Single-input single-output problem")
plt.plot(df1d["x"])
plt.plot(df1d["y"])
plt.plot(residual_1d)
plt.xlabel("Sample")
plt.ylabel("Residual")
plt.show()

In [ ]:
# Run MISO test
df_miso = create_synthetic_data_miso()
res_miso, residual_miso = optimal_linear_combination(
    df_miso, inputs=["x1", "x2", "x3"], output="y", domain="time"
)

print("\nRecovered coefficients:", res_miso.x)
plt.figure()
plt.title("Multiple-input single-output problem")
plt.plot(df_miso["x1"])
plt.plot(df_miso["x2"])
plt.plot(df_miso["x3"])
plt.plot(df_miso["y"])
plt.plot(residual_miso)
plt.xlabel("Sample")
plt.ylabel("Residual")
plt.show()

## Reading the result

Each case prints the recovered coefficients and draws the inputs, the output, and the corrected output on one set of axes. The coefficients should be the negated injected values: 2.5 with one input, and $-1.5$, $0.8$, $-2.0$ with three. The corrected output should look like white noise of standard deviation near the injected 0.1, with no sinusoid and no exponential left in it. A coefficient that is close, on a residual that still shows the input, means the minimizer did not actually reach that coefficient.

The means are removed before the fit, so the corrected output is centered on zero. Over a 10 s record the three inputs are not orthogonal once their means are gone. That does not bias the solution, but an error in one coefficient appears in the others. The tolerance is $10^{-9}$, and `res.success` is whether the minimizer converged. Every trace on the plot shares the axis label "Residual".